# Prediksi Kuat Tekan Beton: Pengaruh Pembersihan Data Duplikat dan Analisis Galat Berdasarkan Umur Beton

**Jenis Masalah:** Regresi Terawasi (*Supervised Regression*) dengan Kuantifikasi Ketidakpastian Konformal  
**Dataset Primer:** *Concrete Compressive Strength* — UCI Machine Learning Repository (Donatur: Prof. I-Cheng Yeh, 1998; $N = 1.030$ sampel, 8 fitur komposisi/umur, 1 target kuat tekan dalam MPa)  
**Tautan Dataset:** https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength

---

### Alur Metodologi Bebas Kebocoran Data (*Zero-Leakage Research Pipeline*)
1. **Pemuatan Data & Audit Forensik Duplikat:** Membedakan duplikat eksak 9 kolom ($25$ baris ekstra $\rightarrow 1.005$ baris unik) dari replikasi identik 8 fitur dengan nilai kuat tekan berbeda ($996$ grup fitur unik pada presisi *float* IEEE 754 `Concrete_Data.xls`, atau $992$ grup unik setelah pembulatan 2 desimal sebagaimana dilaporkan Fakhruddin dkk., 2026).
2. **Exploratory Data Analysis (EDA):** Distribusi target dan fitur, pemeriksaan *outlier* berbasis IQR (mempertahankan umur panjang $91\text{--}365$ hari sebagai domain fisik sah), matriks korelasi, dan pola non-linear hidrasi terhadap umur beton.
3. **Eksperimen Pembersihan Data Duplikat (`raw` vs `exact` vs `median`):** Evaluasi terukur dengan *Group-Aware Split* berdasarkan identitas fitur (`x_group`), *Paired Repeated GroupKFold* ($10 \times 5 = 50$ *fold*), serta uji hipotesis berpasangan (*Paired t-test* & *Wilcoxon signed-rank* dengan koreksi *Holm-Bonferroni*).
4. **Seleksi 10 Kandidat Model & Optimasi Optuna:** Perbandingan *baseline* (`Dummy`, `Ridge`, `SVR`, `Random Forest`) dan keluarga *gradient boosting* (`Gradient Boosting`, `HistGradientBoosting`, `XGBoost`, `LightGBM`, `CatBoost`) di mana pemilihan model pemenang dilakukan murni dari skor *Grouped Cross-Validation* pada himpunan pengembangan.
5. **Analisis Galat Terpilah Berdasarkan Umur Beton (`Age`):** Dekomposisi galat (`MAE`, `RMSE`, `MAPE`, `MBE`/bias, $R^2$, dan cakupan konformal) pada empat fase hidrasi: umur awal (`1–7 hari`), standar (`8–28 hari`), menengah (`29–90 hari`), dan jangka panjang (`91–365 hari`).
6. **Kuantifikasi Ketidakpastian & Interpretabilitas Fisik:** Interval prediksi 90% berbasis *Group-Aware Split Conformal Prediction*, *Permutation Importance*, dan kontribusi lokal/global *TreeSHAP*.

## 0. Import Library & Konfigurasi Reproduksibilitas

In [ ]:
import json
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils import (
    AGE_LABELS,
    FEATURES,
    LABELS,
    TARGET,
    add_features,
    age_groups,
    load_data,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42

for folder in ["figures", "models", "results"]:
    os.makedirs(folder, exist_ok=True)

print("Konfigurasi selesai. RANDOM_STATE =", RANDOM_STATE)

## 1. Pemuatan Dataset & Audit Forensik Data Duplikat

Dataset terdiri atas 8 variabel input dan 1 variabel target (`strength`):

| Variabel | Label Bahasa Indonesia | Satuan | Peran |
|---|---|---|---|
| `cement` | Semen Portland | kg/m³ | Fitur input |
| `slag` | Terak Tanur Tinggi (*Blast Furnace Slag*) | kg/m³ | Fitur input |
| `fly_ash` | Abu Terbang (*Fly Ash*) | kg/m³ | Fitur input |
| `water` | Air Campuran | kg/m³ | Fitur input |
| `superplasticizer` | *Superplasticizer* (Aditif Pereduksi Air) | kg/m³ | Fitur input |
| `coarse_agg` | Agregat Kasar | kg/m³ | Fitur input |
| `fine_agg` | Agregat Halus | kg/m³ | Fitur input |
| `age` | Umur Pengeringan Beton (*Curing Age*) | hari | Fitur input |
| **`strength`** | **Kuat Tekan Beton (*Compressive Strength*)** | **MPa** | **Target** |

In [ ]:
df = load_data("data/Concrete_Data.xls")
print(f"Dimensi dataset mentah : {df.shape[0]} baris x {df.shape[1]} kolom")
print(f"Jumlah fitur input     : {len(FEATURES)} fitur")
print(f"Kolom target           : {TARGET}")
df.head()

In [ ]:
desc = df.describe().T[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]].round(2)
desc

In [ ]:
with open("artifacts/results/report.json", "r", encoding="utf-8") as f:
    report = json.load(f)
dup_audit = report["audit"]
df_rounded2 = df.copy()
df_rounded2[FEATURES] = df_rounded2[FEATURES].round(2)

audit_summary = pd.DataFrame([
    {"Metrik Audit Dataset": "Total sampel mentah (raw)", "Nilai": dup_audit["n_total"], "Keterangan": "Seluruh baris pada Concrete_Data.xls"},
    {"Metrik Audit Dataset": "Baris duplikat eksak (9 kolom identik)", "Nilai": dup_audit["n_exact_duplicates"], "Keterangan": "Baris berulang identik pada 8 fitur + 1 target"},
    {"Metrik Audit Dataset": "Sampel setelah hapus duplikat eksak (exact)", "Nilai": dup_audit["n_unique_exact"], "Keterangan": "1.030 - 25 = 1.005 baris unik (X, y)"},
    {"Metrik Audit Dataset": "Grup 8-fitur unik (presisi float IEEE 754)", "Nilai": dup_audit["n_unique_features"], "Keterangan": f"Terdapat {dup_audit['n_conflicting_feature_groups']} grup berfitur sama namun y berbeda"},
    {"Metrik Audit Dataset": "Grup 8-fitur unik (pembulatan 2 desimal)", "Nilai": int(df_rounded2.drop_duplicates().drop_duplicates(subset=FEATURES).shape[0]), "Keterangan": "Konvensi pembulatan 2 desimal (Fakhruddin dkk., 2026)"},
    {"Metrik Audit Dataset": "Rentang selisih kuat tekan maksimum antar-replikasi", "Nilai": f"{dup_audit['max_target_span_mpa']:.2f} MPa", "Keterangan": "Variasi eksperimental nyata antar benda uji berfitur sama"},
])
audit_summary

**Temuan Kritis Audit Duplikat:**
1. Terdapat **25 baris duplikat eksak** (seluruh 8 fitur dan target `strength` bernilai sama persis), sehingga penghapusan duplikat eksak menyisakan **1.005 baris**.
2. Pada 1.005 baris tersebut, terdapat **996 kombinasi unik 8-fitur** pada presisi *float* mentah `Concrete_Data.xls` (atau **992 kombinasi unik** bila fitur dibulatkan ke 2 angka desimal), yang berarti ada **9 kelompok (18 baris)** dengan komposisi dan umur identik tetapi hasil uji kuat tekan berbeda hingga **33,00 MPa**.
3. Oleh karena itu, pembagian data (*train/test split* dan *cross-validation*) **wajib dikelompokkan berdasarkan identitas 8 fitur (`x_group`)** agar replikasi dari resep campuran yang sama tidak bocor menyeberang antara data latih dan data uji.

## 2. Exploratory Data Analysis (EDA) & Pemeriksaan Domain Fisik

### 2.1 Distribusi Variabel Target (`strength`)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df[TARGET], kde=True, bins=30, ax=ax[0], color="#1d4ed8")
ax[0].set_title("Distribusi Kuat Tekan Beton (MPa)")
ax[0].set_xlabel("Kuat Tekan (MPa)")
sns.boxplot(x=df[TARGET], ax=ax[1], color="#93c5fd")
ax[1].set_title("Boxplot Kuat Tekan Beton")
ax[1].set_xlabel("Kuat Tekan (MPa)")
plt.tight_layout()
plt.savefig("figures/01_distribusi_target.png", dpi=150)
plt.show()
print(f"Rata-rata: {df[TARGET].mean():.2f} MPa | Median: {df[TARGET].median():.2f} MPa | Skewness: {df[TARGET].skew():.3f}")

### 2.2 Distribusi Setiap Fitur Input dan Target

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.ravel(), FEATURES + [TARGET]):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="#0f766e")
    ax.set_title(f"{col} ({LABELS.get(col, 'MPa')})")
plt.tight_layout()
plt.savefig("figures/02_histogram_fitur.png", dpi=150)
plt.show()

### 2.3 Pemeriksaan Outlier Berbasis IQR (Alasan Tidak Membuang Umur Panjang)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), FEATURES + [TARGET]):
    sns.boxplot(x=df[col], ax=ax, color="#e07a5f")
    ax.set_title(col)
plt.tight_layout()
plt.savefig("figures/03_boxplot_fitur.png", dpi=150)
plt.show()

outlier_rows = []
for col in FEATURES + [TARGET]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < low) | (df[col] > high)).sum())
    outlier_rows.append({
        "variabel": col,
        "batas_bawah_iqr": round(low, 2),
        "batas_atas_iqr": round(high, 2),
        "jumlah_di_luar_iqr": n_out,
        "persentase (%)": round(100 * n_out / len(df), 2),
    })
pd.DataFrame(outlier_rows)

**Catatan Metodologis (Koreksi terhadap Pemangkasan Outlier Naif):**  
Sebagian studi terdahulu memangkas baris yang berada di luar pagar IQR pada kolom `age` (yakni beton berumur $>56$ atau $>90$ hari) demi menaikkan skor $R^2$ secara semu. Dalam penelitian ini, **seluruh sampel umur panjang (`91–365 hari`) dipertahankan** karena merupakan pengamatan fisik perkembangan hidrasi beton jangka panjang yang sah dan menjadi objek langsung dari **Analisis Galat Berdasarkan Umur Beton**.

### 2.4 Matriks Korelasi & Hubungan Non-Linear terhadap Umur Beton

In [ ]:
plt.figure(figsize=(9, 7))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriks Korelasi Pearson Fitur Komposisi, Umur, dan Kuat Tekan")
plt.tight_layout()
plt.savefig("figures/04_korelasi.png", dpi=150)
plt.show()
corr[TARGET].drop(TARGET).sort_values(ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for ax, col in zip(axes.ravel(), FEATURES):
    sns.scatterplot(data=df, x=col, y=TARGET, ax=ax, alpha=0.5, s=18, color="#1e3a8a")
    ax.set_title(f"{col} vs {TARGET}")
plt.tight_layout()
plt.savefig("figures/05_scatter_fitur_target.png", dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4.8))
df_eda = df.copy()
df_eda["age_group"] = age_groups(df_eda["age"])
sns.boxplot(data=df_eda, x="age_group", y=TARGET, palette="Blues")
plt.xlabel("Kelompok Umur Beton (hari)")
plt.ylabel("Kuat Tekan Beton (MPa)")
plt.title("Distribusi Kuat Tekan Berdasarkan Fase Umur Hidrasi Beton")
plt.tight_layout()
plt.savefig("figures/06_kuat_tekan_vs_umur.png", dpi=150)
plt.show()

## 3. Eksperimen Pengaruh Pembersihan Data Duplikat (`raw` vs `exact` vs `median`)

Untuk menguji pengaruh pembersihan data duplikat secara adil dan bebas kebocoran (*zero leakage*):
1. Seluruh data terlebih dahulu dipartisi berdasarkan **grup identitas 8 fitur (`x_group`)** menjadi himpunan pengembangan (*development set*, 80%) dan himpunan uji akhir (*holdout test set*, 20% = **200 baris**, **197 grup unik**).
2. Di dalam himpunan pengembangan, ketiga rezim penanganan duplikat dibandingkan pada **50 *fold* validasi silang berpasangan yang identik** (*Paired 10-Repeat 5-Fold GroupKFold*):
   - **`raw` (1.030 baris total):** Mempertahankan seluruh baris asli termasuk duplikat eksak di dalam *fold* latih.
   - **`exact` (1.005 baris):** Menghapus 25 duplikat eksak $(X, y)$ tetapi mempertahankan pengukuran berulang dengan $X$ sama dan $y$ berbeda.
   - **`median` (996 grup fitur unik):** Mengonsolidasikan replikasi berfitur identik pada *fold* latih menggunakan nilai median target $y$.

In [ ]:
with open("artifacts/results/report.json", "r", encoding="utf-8") as f:
    report = json.load(f)

dup_cv_df = pd.DataFrame(report["paired"])[
    ["regime", "n_train_mean", "rmse_mean", "mae_mean", "r2_mean", "delta_rmse_vs_raw", "ci_low", "ci_high", "p_uncorrected", "p_holm"]
].rename(columns={
    "regime": "Rezim Data Latih",
    "n_train_mean": "Rerata Baris Latih/Fold",
    "rmse_mean": "CV RMSE Mean (MPa)",
    "mae_mean": "CV MAE Mean (MPa)",
    "r2_mean": "CV R2 Mean",
    "delta_rmse_vs_raw": "Delta RMSE vs Raw (MPa)",
    "ci_low": "CI 95% Bawah",
    "ci_high": "CI 95% Atas",
    "p_uncorrected": "p-value (Nadeau-Bengio)",
    "p_holm": "p-value (Holm)",
}).round(4)
dup_cv_df

In [ ]:
holdout_regimes_df = pd.DataFrame(report["paired_holdout"]).rename(columns={
    "regime": "Rezim Pelatihan",
    "n_train": "Jumlah Baris Latih",
    "rmse": "Test RMSE (MPa)",
    "mae": "Test MAE (MPa)",
    "r2": "Test R2",
}).round(4)
print("Evaluasi jujur pada Holdout Test Set (200 baris, 200 grup unik):")
holdout_regimes_df[["Rezim Pelatihan", "Jumlah Baris Latih", "Test RMSE (MPa)", "Test MAE (MPa)", "Test R2"]]

**Interpretasi Ilmiah Eksperimen Duplikat:**
- Ketika kebocoran grup (`x_group` *leakage*) dicegah sejak awal menggunakan *GroupKFold* (10 pengulangan × 5 *fold* = 50 evaluasi berpasangan), perbedaan RMSE lintas ketiga rezim pada CV berada dalam rentang yang sangat sempit (`4,541–4,574 MPa`, $p_{\text{Holm}} = 1,000$ dengan koreksi variansi Nadeau–Bengio).
- Namun pada evaluasi *holdout test set* ($n = 200$) dengan model pemenang CatBoost, rezim **`exact_dedup` (1.005 baris)** menghasilkan akurasi uji terbaik (**Test RMSE = 3,5307 MPa**, **MAE = 2,6086 MPa**, **$R^2 = 0,9479$**) dibandingkan `raw` (`RMSE = 3,6126 MPa`) maupun `feature_consolidated` (`RMSE = 3,5464 MPa`), sekaligus menjaga keutuhan variasi eksperimental laboratorium.

## 4. Perbandingan 10 Kandidat Model & Optimasi Hyperparameter Optuna

Seluruh kandidat dievaluasi menggunakan *5-Fold GroupKFold Cross-Validation* pada himpunan pelatihan (`exact_dedup`, $n = 642$), mencakup:
- **Model Baseline & Regresi Terstruktur:** `Dummy mean`, `Ridge`, `SVR`, `Random Forest`.
- **Keluarga Gradient Boosting Modern:** `Gradient Boosting`, `LightGBM`, `XGBoost`, `CatBoost`, serta kandidat teroptimasi `XGBoost + Optuna` dan `CatBoost + Optuna` (menggunakan *inner GroupKFold* murni pada data latih).

In [ ]:
model_cv_df = pd.DataFrame(report["selection"])[
    ["model", "rmse_mean", "rmse_std", "mae_mean", "r2_mean"]
].rename(columns={
    "model": "Kandidat Model",
    "rmse_mean": "CV RMSE Mean (MPa)",
    "rmse_std": "CV RMSE Std (MPa)",
    "mae_mean": "CV MAE Mean (MPa)",
    "r2_mean": "CV R2 Mean",
}).round(4)
model_cv_df

In [ ]:
holdout_models_df = pd.DataFrame(report["holdout_models"])[
    ["model", "rmse", "mae", "r2"]
].rename(columns={
    "model": "Model",
    "rmse": "Test RMSE (MPa)",
    "mae": "Test MAE (MPa)",
    "r2": "Test R2",
}).sort_values("Test RMSE (MPa)").reset_index(drop=True).round(4)

holdout_models_df = holdout_models_df.merge(
    model_cv_df[["Kandidat Model", "CV RMSE Mean (MPa)", "CV R2 Mean"]],
    left_on="Model",
    right_on="Kandidat Model",
    how="left",
).drop(columns=["Kandidat Model"])

# Simpan ringkasan kompatibilitas ke results/perbandingan_model.csv
holdout_models_df.to_csv("results/perbandingan_model.csv", index=False)
holdout_models_df

In [ ]:
# Visualisasi perbandingan metrik pada Holdout Test Set (tanpa Dummy baseline)
plot_df = holdout_models_df[holdout_models_df["Model"] != "Dummy mean"].copy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric, col in zip(
    axes,
    ["Test MAE (MPa)", "Test RMSE (MPa)", "Test R2"],
    ["#2563eb", "#d97706", "#059669"],
):
    sns.barplot(data=plot_df, x="Model", y=metric, ax=ax, color=col)
    ax.set_title(metric)
    ax.set_xlabel("")
    for i, v in enumerate(plot_df[metric]):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    ax.tick_params(axis="x", rotation=28)
plt.tight_layout()
plt.savefig("figures/07_perbandingan_metrik.png", dpi=150)
plt.show()

In [ ]:
# Plot Aktual vs Prediksi dan Distribusi Residual untuk Model Pemenang (CatBoost)
test_preds = pd.DataFrame(report["scatter"])
q90 = float(report["conformal"]["q"])
test_preds["residual_mpa"] = test_preds["predicted"] - test_preds["actual"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

lo, hi = test_preds["actual"].min(), test_preds["actual"].max()
axes[0].scatter(test_preds["actual"], test_preds["predicted"], alpha=0.65, s=24, color="#1d4ed8")
axes[0].plot([lo, hi], [lo, hi], "r--", lw=1.4, label="Garis Ideal (y = y_hat)")
axes[0].fill_between(
    np.sort(test_preds["actual"]),
    np.sort(test_preds["actual"]) - q90,
    np.sort(test_preds["actual"]) + q90,
    color="#93c5fd",
    alpha=0.25,
    label=f"Pita Konformal 90% (±{q90:.2f} MPa)",
)
axes[0].set_title(f"Aktual vs Prediksi — {report['winner']} (Test R² = {report['test_metrics']['r2']:.3f})")
axes[0].set_xlabel("Kuat Tekan Aktual (MPa)")
axes[0].set_ylabel("Kuat Tekan Prediksi (MPa)")
axes[0].legend()

sns.histplot(test_preds["residual_mpa"], kde=True, bins=25, ax=axes[1], color="#6d28d9")
axes[1].axvline(0, color="k", ls="--", lw=1.2)
axes[1].set_title(f"Distribusi Residual Uji ({report['winner']})")
axes[1].set_xlabel("Residual = Prediksi - Aktual (MPa)")
plt.tight_layout()
plt.savefig("figures/08_aktual_vs_prediksi.png", dpi=150)
plt.savefig("figures/09_residual.png", dpi=150)
plt.show()

In [ ]:
# Perbandingan CV R2 vs Test R2 Lintas Kandidat Model
fig, ax = plt.subplots(figsize=(9.5, 4.5))
x_idx = np.arange(len(plot_df))
width = 0.36
ax.bar(x_idx - width / 2, plot_df["CV R2 Mean"], width, label="CV R² (5-Fold GroupKFold)", color="#93c5fd")
ax.bar(x_idx + width / 2, plot_df["Test R2"], width, label="Test R² (Holdout)", color="#1d4ed8")
ax.set_xticks(x_idx)
ax.set_xticklabels(plot_df["Model"], rotation=25, ha="right")
ax.set_ylim(0.75, 1.00)
ax.set_ylabel("Koefisien Determinasi (R²)")
ax.set_title("Perbandingan CV R² vs Holdout Test R² Lintas Kandidat Model")
ax.legend()
plt.tight_layout()
plt.savefig("figures/10_train_vs_test.png", dpi=150)
plt.show()

**Keputusan Pemilihan Model Terbaik:**  
Berdasarkan protokol *5-Fold GroupKFold Cross-Validation* pada data latih (tanpa mengintip data kalibrasi maupun data uji), **CatBoost** terpilih sebagai model terbaik (`CV RMSE = 4,3767 ± 0,3995 MPa`, `CV MAE = 3,0050 MPa`, `CV R² = 0,9283`). Pada pengujian akhir (*holdout test set*, $n = 200$), **CatBoost** mencapai:
- **RMSE Uji Akhir:** `3,5307 MPa`
- **MAE Uji Akhir:** `2,6086 MPa`
- **Koefisien Determinasi ($R^2$):** `0,9479` (menjelaskan 94,79% variasi kuat tekan beton)

## 5. Analisis Galat Berdasarkan Kelompok Umur Beton (`Age`)

Untuk menjawab bagian kedua dari judul penelitian, galat pada *holdout test set* ($n = 200$) dianalisis secara terpilah ke dalam empat rezim umur hidrasi beton:
1. **`1–7 hari` (Umur Awal):** Laju reaksi hidrasi $C_3S$ sangat cepat; sensitivitas terhadap rasio air/pengikat dan *superplasticizer* sangat tinggi.
2. **`8–28 hari` (Umur Standar Desain):** Fase pematangan menuju kuat tekan karakteristik 28 hari.
3. **`29–90 hari` (Umur Menengah):** Reaksi pozzolanik sekunder dari *fly ash* dan *slag* mulai mendominasi.
4. **`91–365 hari` (Umur Jangka Panjang):** Pematangan lanjut dengan kepadatan sampel yang lebih jarang di dataset.

In [ ]:
age_df = pd.DataFrame(report["age_metrics"])[
    ["group", "n", "n_groups", "mae", "mae_ci_low", "mae_ci_high", "rmse", "bias", "coverage"]
].rename(columns={
    "group": "Kelompok Umur (hari)",
    "n": "Jumlah Uji (n)",
    "n_groups": "Grup Unik",
    "mae": "MAE (MPa)",
    "mae_ci_low": "MAE CI 95% Bawah",
    "mae_ci_high": "MAE CI 95% Atas",
    "rmse": "RMSE (MPa)",
    "bias": "Bias / MBE (MPa)",
    "coverage": "Cakupan Konformal 90%",
}).round(4)
age_df

**Interpretasi Galat Berdasarkan Umur Beton:**
- **Umur Standar (`8–28 hari`, $n = 94$):** Merupakan populasi terbesar di data uji dan mencatat galat absolut tertinggi (`MAE = 2,9087 MPa`, `RMSE = 3,8890 MPa`) karena tingginya keragaman desain campuran komersial pada umur pengujian standar 14 dan 28 hari.
- **Umur Awal (`1–7 hari`, $n = 52$):** Mencapai `MAE = 2,6405 MPa` (`CI 95%: 2,0383–3,3424 MPa`), `RMSE = 3,5695 MPa`, dan cakupan konformal **92,31%**.
- **Umur Menengah (`29–90 hari`, $n = 29$) & Jangka Panjang (`91–365 hari`, $n = 25$):** Menunjukkan penurunan galat absolut (`MAE = 2,3148 MPa` dan `1,7543 MPa`; `RMSE = 3,0467 MPa` dan `2,3641 MPa`) dengan cakupan konformal **93,10%** dan **100,0%**, serta bias sedikit negatif (`-0,6339 MPa`) pada `91–365 hari`.

## 6. Interpretabilitas Model (*Permutation Importance* & *TreeSHAP*) serta Interval Konformal

In [ ]:
perm_df = pd.DataFrame(report["importance"]).rename(columns={
    "feature": "Fitur",
    "importance": "Penurunan RMSE Saat Diacak (MPa)",
    "std": "Std (MPa)",
}).round(4)
perm_df["Nama Fitur"] = perm_df["Fitur"].map(lambda k: LABELS.get(k, k))

shap_df = pd.DataFrame(report["shap"]["features"]).rename(columns={
    "feature": "Fitur",
    "mean_abs_shap": "Mean |SHAP| (MPa)",
}).round(4)
shap_df["Nama Fitur"] = shap_df["Fitur"].map(lambda k: LABELS.get(k, k))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.barplot(data=perm_df, x="Penurunan RMSE Saat Diacak (MPa)", y="Nama Fitur", ax=axes[0], color="#0f766e")
axes[0].set_title("Permutation Importance pada Data Uji (ΔRMSE MPa)")

sns.barplot(data=shap_df, x="Mean |SHAP| (MPa)", y="Nama Fitur", ax=axes[1], color="#b45309")
axes[1].set_title("Global TreeSHAP Importance (Mean |SHAP| MPa)")
plt.tight_layout()
plt.savefig("figures/11_permutation_importance.png", dpi=150)
plt.savefig("figures/12_feature_importance.png", dpi=150)
plt.show()

perm_df[["Fitur", "Nama Fitur", "Penurunan RMSE Saat Diacak (MPa)", "Std (MPa)"]]

## 7. Verifikasi Bundel Model Produksi & Simulasi Prediksi

Model pemenang beserta parameter kalibrasi konformal dan detektor *Out-of-Distribution* (Mahalanobis) disimpan di `artifacts/models/model_bundle.joblib` (serta salinan kompatibilitas di `models/best_model.joblib` dan `models/metadata.json`) untuk digunakan langsung oleh aplikasi web FastAPI (`app.py`).

In [ ]:
with open("models/metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_model": report["winner"],
        "features": FEATURES,
        "metrics_test": report["test_metrics"],
        "conformal": report["conformal"],
        "n_rows": report["audit"]["n_unique_exact"],
    }, f, indent=2)

sample_row = report["scatter"][0]
q90 = float(report["conformal"]["q"])
print(f"Model Aktif         : {report['winner']}")
print(f"Contoh Aktual Uji   : {sample_row['actual']:.2f} MPa (Umur {sample_row['age']:.0f} hari, Grup {sample_row['group']})")
print(f"Prediksi Titik      : {sample_row['predicted']:.2f} MPa")
print(f"Interval Konformal  : [{sample_row['predicted'] - q90:.2f}, {sample_row['predicted'] + q90:.2f}] MPa (±{q90:.2f} MPa, Tingkat Kepercayaan 90%)")

## 8. Kesimpulan

1. **Pengaruh Pembersihan Data Duplikat:** Audit pada dataset UCI Concrete ($1.030$ baris) menemukan **25 baris duplikat eksak** ($1.005$ baris unik) dan **996 kombinasi unik 8-fitur** pada presisi *float* mentah (`992` pada pembulatan 2 desimal), dengan 9 kelompok replikasi yang memiliki selisih kuat tekan hingga $33,00\text{ MPa}$. Dengan protokol *Group-Aware Split* berdasarkan `x_group`, penghapusan duplikat eksak (`exact_dedup`) memberikan generalisasi *holdout test* terbaik (**RMSE = 3,5307 MPa**, **MAE = 2,6086 MPa**, **$R^2$ = 0,9479**) sekaligus mempertahankan variasi eksperimental laboratorium.
2. **Performa Model Terbaik:** Dari 10 kandidat model yang dievaluasi bebas kebocoran data, **CatBoost** unggul konsisten baik pada *5-Fold GroupKFold CV* (`RMSE = 4,3767 MPa`, `R² = 0,9283`) maupun pada *holdout test set* ($n = 200$, `RMSE = 3,5307 MPa`, `MAE = 2,6086 MPa`, `R² = 0,9479`).
3. **Analisis Galat Berdasarkan Umur Beton:** Galat absolut tertinggi terjadi pada fase umur desain standar `8–28 hari` (`MAE = 2,9087 MPa`, `RMSE = 3,8890 MPa`), diikuti umur awal `1–7 hari` (`MAE = 2,6405 MPa`, `RMSE = 3,5695 MPa`), sedangkan umur menengah `29–90 hari` (`MAE = 2,3148 MPa`) dan jangka panjang `91–365 hari` (`MAE = 1,7543 MPa`) memiliki galat absolut yang lebih rendah.
4. **Faktor Dominan & Ketidakpastian:** *Permutation Importance* mengonfirmasi bahwa **umur beton (`age`, $\Delta\text{RMSE} = 11,31\text{ MPa}$)** dan **semen (`cement`, $\Delta\text{RMSE} = 11,20\text{ MPa}$)** adalah penentu utama kuat tekan di tingkat fitur input, sementara *TreeSHAP* menunjukkan peran sentral fitur rekayasa domain **rasio air/pengikat (`w_b_ratio`, $\text{Mean }|\text{SHAP}| = 5,48\text{ MPa}$)** dan **transformasi logaritmik umur (`log_age`, $4,59\text{ MPa}$)**. *Group-Aware Split Conformal Prediction* menghasilkan interval prediksi $\pm 5,86\text{ MPa}$ dengan cakupan aktual tepat **90,0%** pada data uji.